# Milestone 3: Evaluate, track and monitor
**Project: Northwind Knowledge Assistant** | **Presenter: Nagarjuna Naidu** | Day 4, about 30 minutes

Goal: compare two chunk sizes with MLflow, watch for data drift (PSI), and decide between two versions with an A/B test.
No LLM is needed in this milestone.

## 1. Compare two configurations and log them in MLflow
Chunks of 60 words vs 150 words. Recall@3 is measured on the golden questions in `data/eval_queries.json`.

In [ ]:
import os, json, logging, warnings
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore")
import mlflow
logging.getLogger("mlflow").setLevel(logging.ERROR)
from nka.config import ARTIFACTS
from nka.evalkit import evaluate_retrieval

mlflow.set_tracking_uri(f"sqlite:///{ARTIFACTS / 'mlflow.db'}")
name = "nka-retrieval"
exp = mlflow.get_experiment_by_name(name) or mlflow.get_experiment(mlflow.create_experiment(name, artifact_location=str(ARTIFACTS / "mlruns")))
for words in (60, 150):
    res = evaluate_retrieval(chunk_words=words)
    with mlflow.start_run(experiment_id=exp.experiment_id, run_name=f"chunk{words}"):
        mlflow.log_param("chunk_words", words)
        mlflow.log_metrics({"recall_overall": res["overall"], "chunks": res["chunks"],
                            **{f"recall_{t}": v for t, v in res["recall_at_k"].items()}})
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=["metrics.recall_overall DESC", "start_time DESC"], max_results=20).drop_duplicates("params.chunk_words")
print(runs[["tags.mlflow.runName", "params.chunk_words", "metrics.recall_overall", "metrics.recall_keyword"]].to_string(index=False))
best = int(runs.iloc[0]["params.chunk_words"])
json.dump({"chunk_words": best}, open(ARTIFACTS / "best_config.json", "w"))
print("best chunk size:", best, "-> saved to artifacts/best_config.json")

## 2. Monitor: has the data drifted?
PSI compares today's traffic with a reference. **Below 0.1** stable, **0.1 to 0.25** watch, **above 0.25** drifted.

In [ ]:
import matplotlib.pyplot as plt
from nka.monitor import simulate_traffic, psi

ref, now = simulate_traffic(500), simulate_traffic(500, drift=True, seed=1)
feats = ["query_length", "top_score", "latency_ms"]
scores = {f: psi(ref[f], now[f]) for f in feats}
for f, v in scores.items():
    print(f"{f:13} PSI {v:.2f}  {'drifted' if v > 0.25 else 'watch' if v > 0.1 else 'stable'}")
plt.figure(figsize=(5, 2.5)); plt.bar(scores.keys(), scores.values(), color="#4c78a8")
plt.axhline(0.1, ls="--", c="orange"); plt.axhline(0.25, ls="--", c="red"); plt.ylabel("PSI"); plt.show()

## 3. A/B test: is version B really better?
Simulated thumbs-up counts: A is served on the drifted traffic, B (re-tuned) on normal traffic.

In [ ]:
from nka.monitor import ab_test

a, b = now["thumbs_up"], ref["thumbs_up"]
print("A thumbs-up:", a.sum(), "of", len(a), "| B thumbs-up:", b.sum(), "of", len(b))
print({k: (v if isinstance(v, str) else float(v)) for k, v in ab_test(a.sum(), len(a), b.sum(), len(b)).items()})

## 4. YOUR TASK
A golden list is a set of questions with the document that must be found. Add one question of your own (with its expected doc id, see `data/corpus`) and re-evaluate.

In [ ]:
from nka.index import get_index

extra = [dict(query="how much time off do I get when my baby is born", doc="D01")]
# TODO: extra.append(dict(query="your question", doc="D.."))
idx = get_index()
hits = [any(h["doc_id"] == q["doc"] for h in idx.search(q["query"], k=3, role="hr_partner")) for q in extra]
print(f"golden list: {sum(hits)}/{len(hits)} found in the top 3")

**Canvas section 6:** write one line about the metric you will track and the threshold that triggers action.